## 8. Automatic selection of the final six, decoding and comparison with the exact optimum

# STAGE 3 – P(at least 3 of the 6 selected customers subscribe), quantum-circuit estimation

**Circuit:** one qubit per selected customer, `RY(θ_k)|0⟩ = cos(θ_k/2)|0⟩ + sin(θ_k/2)|1⟩`, `θ_k = 2·arcsin(√p_k)` ⇒ P(q_k = 1) = p_k; independence ⇒ product state (no entangling gates). `1` = subscribes. **Success ⇔ Hamming weight ≥ 3 (42 of 64 basis states).**

**Customer-to-qubit order:** ascending `customer_id` → q0, q1, …, q5.
**Bit order (Qiskit little-endian):** printed `b5 b4 b3 b2 b1 b0`, **q0 = right-most (LSB)**, integer index = Σ b_k 2^k. Big-endian reference (`b0 b1 … b5`) is printed alongside.

**Benchmarks / regimes:** (1) exact Poisson-binomial (DP + 64-state enumeration) · (2) ideal statevector · (3) noiseless finite-shot circuit (100 000 shots) · (4) classical Monte-Carlo Bernoulli (100 000 trials) · (5) noisy circuit (single-qubit depolarising gate noise + asymmetric readout error) · (6) mitigated circuit (tensor-product readout-calibration inversion + Richardson zero-noise extrapolation) · (7) MLAE amplitude estimation. The best estimator is chosen automatically by RMSE against the exact value.

In [13]:
import numpy as np
import pandas as pd

sel = pd.DataFrame({
    "customer_id": ["C0297", "C0073", "C0107", "C0017", "C0311", "C0281"],
    "probability": [0.45, 0.62, 0.58, 0.35, 0.70, 0.51]  # replace with your exact Stage 1 predicted probabilities
})

In [14]:
# ---------- qubit map (ascending customer_id) ----------
order = sel.sort_values("customer_id").reset_index(drop=True)
cust = order["customer_id"].tolist(); p = order["probability"].to_numpy(float); n = len(p)
theta = 2*np.arcsin(np.sqrt(p)); dim = 2**n
SHOTS, REPS = 100_000, 30
def bs(i):    return format(i, f"0{n}b")            # little-endian print: b5 b4 b3 b2 b1 b0
def bs_be(i): return bs(i)[::-1]                    # big-endian reference: b0 b1 b2 b3 b4 b5
popc = np.array([bin(s).count("1") for s in range(dim)]); good = popc >= 3
assert good.sum() == 42
print(pd.DataFrame({"qubit": [f"q{k}" for k in range(n)], "customer_id": cust, "p_stage1": p, "theta_RY(rad)": theta}).to_string(index=False))
print("example: index 5 -> little-endian", bs(5), "| big-endian", bs_be(5), "(q0=1,q2=1)")

# ---------- 1) exact classical ----------
def exact_pb(p, kmin=3):
    d = np.zeros(len(p)+1); d[0] = 1.0
    for pi in p: d[1:] = d[1:]*(1-pi) + d[:-1]*pi; d[0] *= (1-pi)
    return d[kmin:].sum(), d
P_exact, count_dist = exact_pb(p)
P_enum = sum(np.prod([p[k] if (s >> k) & 1 else 1-p[k] for k in range(n)]) for s in range(dim) if popc[s] >= 3)
assert abs(P_exact - P_enum) < 1e-12
print(f"\nEXACT P(at least 3) = {P_exact:.10f}")

qubit customer_id  p_stage1  theta_RY(rad)
   q0       C0017      0.35       1.266104
   q1       C0073      0.62       1.813162
   q2       C0107      0.58       1.731487
   q3       C0281      0.51       1.590798
   q4       C0297      0.45       1.470629
   q5       C0311      0.70       1.982313
example: index 5 -> little-endian 000101 | big-endian 101000 (q0=1,q2=1)

EXACT P(at least 3) = 0.7257765900


In [15]:
pip install qiskit

Note: you may need to restart the kernel to use updated packages.


In [16]:
# ---------- circuit layer (Qiskit if installed, else equivalent built-in simulator) ----------
try:
    from qiskit import QuantumCircuit, transpile
    from qiskit.quantum_info import Statevector
    try:    from qiskit_aer import AerSimulator
    except Exception: AerSimulator = None
    HAVE_QISKIT = True
except Exception:
    HAVE_QISKIT, AerSimulator = False, None
USE_AER = HAVE_QISKIT and AerSimulator is not None
print("Qiskit:", HAVE_QISKIT, "| Aer:", USE_AER)

def ry(t): return np.array([[np.cos(t/2), -np.sin(t/2)], [np.sin(t/2), np.cos(t/2)]])
def build_A():
    U = np.array([[1.0]])
    for k in range(n): U = np.kron(ry(theta[k]), U)        # q0 ends up least-significant
    return U
A = build_A(); e0 = np.zeros(dim); e0[0] = 1
psi = A @ e0; probs_ideal = np.abs(psi)**2                  # 2) ideal statevector

def qiskit_circuit(measure=False):
    qc = QuantumCircuit(n)
    for k in range(n): qc.ry(float(theta[k]), k)
    if measure: qc.measure_all()
    return qc
if HAVE_QISKIT:
    assert np.allclose(Statevector(qiskit_circuit()).probabilities(), probs_ideal, atol=1e-10)
    print("Qiskit statevector == built-in simulator (little-endian verified)")

P_sv = float(probs_ideal[good].sum()); assert abs(P_sv - P_exact) < 1e-10
pd.DataFrame({"little-endian b5..b0": [bs(i) for i in np.argsort(-probs_ideal)[:6]],
              "big-endian b0..b5":   [bs_be(i) for i in np.argsort(-probs_ideal)[:6]],
              "prob": np.sort(probs_ideal)[::-1][:6]})

Qiskit: True | Aer: False
Qiskit statevector == built-in simulator (little-endian verified)


,little-endian b5..b0,big-endian b0..b5,prob
0,101110,011101,0.045895
1,100110,011001,0.044095
2,111110,011111,0.037550
3,110110,011011,0.036078
4,101010,010101,0.033234
5,100010,010001,0.031931


In [17]:
# ---------- 3) noiseless finite shots, 4) classical Monte-Carlo ----------
def draw(probs, shots, r):
    return np.bincount(r.choice(dim, shots, p=probs/probs.sum()), minlength=dim)
def est_ideal_shots(seed):
    if USE_AER:
        sim = AerSimulator(seed_simulator=int(seed))
        c = sim.run(transpile(qiskit_circuit(True), sim), shots=SHOTS).result().get_counts()
        return sum(v for b, v in c.items() if b.replace(" ", "").count("1") >= 3) / SHOTS
    return draw(probs_ideal, SHOTS, np.random.default_rng(seed))[good].sum() / SHOTS
def est_classical_mc(seed):
    r = np.random.default_rng(seed); X = r.random((SHOTS, n)) < p
    return float((X.sum(1) >= 3).mean())

# ---------- 5) noise model ----------
EPS_GATE = 0.02                                   # single-qubit depolarising prob per RY gate
R01, R10 = 0.02, 0.06                             # asymmetric readout: P(read1|0), P(read0|1)
def noisy_probs(scale=1):
    """product of per-qubit density matrices; scale = gate-folding factor (RY (RY† RY)^m) -> eps_eff=1-(1-eps)^scale"""
    eps = 1 - (1-EPS_GATE)**scale; out = np.array([1.0])
    for k in range(n):
        rho = np.outer(ry(theta[k])[:, 0], ry(theta[k])[:, 0])
        rho = (1-eps)*rho + eps*np.eye(2)/2                     # depolarising channel
        pk = np.real(np.diag(rho)); out = np.kron(pk, out)      # q0 stays least-significant
    return out
CONF1 = np.array([[1-R01, R10], [R01, 1-R10]])                  # columns = true state
def readout(pr):                                                # apply asymmetric readout error on every qubit
    t = pr.reshape((2,)*n)
    for ax in range(n): t = np.moveaxis(np.tensordot(CONF1, t, axes=([1], [ax])), 0, ax)
    return t.reshape(-1)
def noisy_counts(scale, shots, r): return draw(readout(noisy_probs(scale)), shots, r)

# ---------- 6) mitigation: calibration-matrix inversion + Richardson ZNE ----------
def calibrate(r, cal_shots=20_000):
    """estimate per-qubit confusion matrices from |0…0> and |1…1> calibration circuits"""
    c0 = draw(readout(np.eye(dim)[0]), cal_shots, r); c1 = draw(readout(np.eye(dim)[-1]), cal_shots, r)
    Ms = []
    for k in range(n):
        r01 = sum(c0[s] for s in range(dim) if (s >> k) & 1) / cal_shots
        r10 = sum(c1[s] for s in range(dim) if not (s >> k) & 1) / cal_shots
        Ms.append(np.array([[1-r01, r10], [r01, 1-r10]]))
    return Ms
def mitigate_readout(counts, Ms):
    t = (counts / counts.sum()).reshape((2,)*n)
    for k in range(n):
        ax = n-1-k                                              # reshape axis of qubit k (little-endian)
        t = np.moveaxis(np.tensordot(np.linalg.inv(Ms[k]), t, axes=([1], [ax])), 0, ax)
    m = np.clip(t.reshape(-1), 0, None); return m / m.sum()
SCALES = [1, 3, 5]; RICH = np.array([15/8, -5/4, 3/8])           # Richardson (quadratic) weights at x=0, sum=1
def est_noisy_raw(seed):  return noisy_counts(1, SHOTS, np.random.default_rng(seed))[good].sum() / SHOTS
def est_noisy_ro(seed):
    r = np.random.default_rng(seed); Ms = calibrate(r)
    return float(mitigate_readout(noisy_counts(1, SHOTS, r), Ms)[good].sum())
def est_noisy_ro_zne(seed):
    r = np.random.default_rng(seed); Ms = calibrate(r)
    vals = [mitigate_readout(noisy_counts(c, SHOTS, r), Ms)[good].sum() for c in SCALES]
    return float(RICH @ np.array(vals))
print("noisy (scale 1) exact-limit P(>=3):", round(float(readout(noisy_probs(1))[good].sum()), 6), "vs exact", round(P_exact, 6))

noisy (scale 1) exact-limit P(>=3): 0.682202 vs exact 0.725777


In [18]:
# ---------- 7) MLAE (noiseless Grover amplification of the same circuit) ----------
S0 = np.eye(dim); S0[0, 0] = -1
Schi = np.diag(np.where(good, -1.0, 1.0)); Q = A @ S0 @ A.T @ Schi
KS = [0, 1, 2, 4, 8, 16]; NSH = SHOTS // len(KS)
pk_good = []; vec = psi.copy(); kc = 0
for k in KS:
    while kc < k: vec = Q @ vec; kc += 1
    pk_good.append(float((np.abs(vec)**2)[good].sum()))
pk_good = np.array(pk_good); grid = np.linspace(1e-6, np.pi/2-1e-6, 40001)
def est_mlae(seed):
    h = np.random.default_rng(seed).binomial(NSH, pk_good); ll = np.zeros_like(grid)
    for k, hk in zip(KS, h):
        s2 = np.clip(np.sin((2*k+1)*grid)**2, 1e-12, 1-1e-12); ll += hk*np.log(s2) + (NSH-hk)*np.log(1-s2)
    return float(np.sin(grid[np.argmax(ll)])**2)

# ---------- automatic comparison / selection ----------
methods = {   # name: (function, group, is_quantum)
 "Noiseless circuit shots (100k)":         (est_ideal_shots,  "ideal device",    True),
 "MLAE amplitude estimation (≈100k shots)":(est_mlae,         "ideal device",    True),
 "Classical Monte-Carlo (100k trials)":    (est_classical_mc, "classical",       False),
 "Noisy circuit (raw)":                    (est_noisy_raw,    "noisy device",    True),
 "Noisy + readout-mitigated":              (est_noisy_ro,     "noisy device",    True),
 "Noisy + readout-mitigated + ZNE":        (est_noisy_ro_zne, "noisy device",    True)}
reps = REPS if not USE_AER else 8
rows = []
for name, (f, grp, q) in methods.items():
    est = np.array([f(5000+i) for i in range(reps)]); err = est - P_exact
    rows.append(dict(method=name, regime=grp, quantum=q, mean=est.mean(), std=est.std(ddof=1),
                     bias=err.mean(), RMSE=np.sqrt((err**2).mean())))
cmp_df = pd.DataFrame(rows).set_index("method"); print(cmp_df.round(6).to_string())

best_noisy = cmp_df[cmp_df.regime == "noisy device"]["RMSE"].idxmin()
best_ideal = cmp_df[(cmp_df.regime == "ideal device")]["RMSE"].idxmin()
print("\nBest estimator on a NOISY device :", best_noisy)
print("Best estimator on an IDEAL device:", best_ideal)

                                               regime  quantum      mean       std      bias      RMSE
method                                                                                                
Noiseless circuit shots (100k)           ideal device     True  0.725622  0.001687 -0.000155  0.001666
MLAE amplitude estimation (≈100k shots)  ideal device     True  0.725788  0.000087  0.000012  0.000086
Classical Monte-Carlo (100k trials)         classical    False  0.725662  0.001540 -0.000115  0.001518
Noisy circuit (raw)                      noisy device     True  0.681950  0.001459 -0.043827  0.043850
Noisy + readout-mitigated                noisy device     True  0.724050  0.001425 -0.001726  0.002224
Noisy + readout-mitigated + ZNE          noisy device     True  0.725877  0.003112  0.000100  0.003061

Best estimator on a NOISY device : Noisy + readout-mitigated
Best estimator on an IDEAL device: MLAE amplitude estimation (≈100k shots)


In [19]:
# ---------- final reported numbers ----------
final_fn = methods[best_noisy][0]; P_final = final_fn(424242)
P_ideal_best = methods[best_ideal][0](424243)
final = pd.DataFrame([
  ["Exact Poisson-binomial", P_exact, 0.0],
  ["Ideal statevector (circuit)", P_sv, P_sv - P_exact],
  [f"Ideal-device best: {best_ideal}", P_ideal_best, P_ideal_best - P_exact],
  [f"FINAL (noisy-device best): {best_noisy}", P_final, P_final - P_exact]], columns=["method", "P(at least 3)", "error vs exact"])
print(final.to_string(index=False))
dist_c = np.array([probs_ideal[popc == j].sum() for j in range(n+1)])
print(pd.DataFrame({"#subscribers": range(n+1), "exact": count_dist, "circuit": dist_c}).round(6).to_string(index=False))
pd.DataFrame({"qubit": [f"q{k}" for k in range(n)], "customer_id": cust, "p_stage1": p, "theta_RY": theta}).to_csv("stage_3_qubit_map.csv", index=False)
cmp_df.to_csv("stage_3_method_comparison.csv"); final.to_csv("stage_3_report.csv", index=False)
print(f"\nFINAL ANSWER: P(at least 3 of {n} subscribe) ≈ {P_final:.6f} (exact {P_exact:.6f}, |error| {abs(P_final-P_exact):.2e})")

                                                    method  P(at least 3)  error vs exact
                                    Exact Poisson-binomial       0.725777    0.000000e+00
                               Ideal statevector (circuit)       0.725777   -3.330669e-16
Ideal-device best: MLAE amplitude estimation (≈100k shots)       0.725910    1.330398e-04
      FINAL (noisy-device best): Noisy + readout-mitigated       0.724817   -9.593015e-04
 #subscribers    exact  circuit
            0 0.008387 0.008387
            1 0.064946 0.064946
            2 0.200890 0.200890
            3 0.317660 0.317660
            4 0.270508 0.270508
            5 0.117389 0.117389
            6 0.020219 0.020219

FINAL ANSWER: P(at least 3 of 6 subscribe) ≈ 0.724817 (exact 0.725777, |error| 9.59e-04)
